# START HERE · 설치부터 결과 시험까지
이 노트북 하나를 위에서부터 실행하면 설치 → 맵 → 테스트 → 학습 → 평가 → 그래프·영상·보고서 → 모델 시험으로 이어집니다. 전체 프로젝트 ZIP을 먼저 업로드합니다.

**제공 모델만 시험:** 설치 셀을 실행한 뒤 마지막 `06 · 학습된 모델 시험하기`로 이동하세요. 새 학습은 `MODE`, `TRAIN_DIR`, `RESUME`를 확인한 뒤 실행하세요.

# 00 · 설치 및 연결 확인
전체 프로젝트 ZIP을 업로드하고 SUMO·PyTorch 환경을 확인합니다. GPU는 필수가 아닙니다. 기존 PyTorch가 있으면 유지합니다.

In [ ]:
from pathlib import Path
import os,sys,zipfile,subprocess
if 'google.colab' in sys.modules or Path('/content').exists():
    from google.colab import files
    if not Path('/content/T_Junction_RL/REVISION_V2').exists():
        uploaded = files.upload()
        filename = next((x for x in uploaded if x.endswith('.zip')), None)
        if filename is None: raise ValueError('전체 프로젝트 ZIP을 선택하세요.')
        with zipfile.ZipFile(filename) as archive:
            for member in archive.infolist():
                destination=(Path('/content')/member.filename).resolve()
                if not destination.is_relative_to(Path('/content')): raise ValueError('잘못된 ZIP 경로')
            archive.extractall('/content')

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
subprocess.run(['bash','scripts/setup_colab.sh'], check=True)

## 선택: Drive 연결
체크포인트는 실행 중 로컬에 쓰고 완료 체크포인트와 로그를 Drive에 주기적으로 복사합니다.

In [ ]:
MOUNT_DRIVE = True
DRIVE_RUNS = None
if MOUNT_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RUNS = Path('/content/drive/MyDrive/T_Junction_RL/runs')
    DRIVE_RUNS.mkdir(parents=True,exist_ok=True)
print('지속 보관 위치:', DRIVE_RUNS)

In [ ]:
from maps.generate_map import generate
from src.traffic_generator import generate_traffic
from src.evaluate import run_episode
from src.utils import load_yaml
print('맵:', generate())
cfg=load_yaml('base.yaml');cfg['simulation']['horizon_seconds']=30
route=generate_traffic('medium',1000,30,directory=Path('outputs/setup_check/routes'))
result=run_episode(cfg,route,1000,'fixed_time',Path('outputs/setup_check/run'))
print('SUMO + TraCI simulationStep 성공:', result)

# 01 · T자형 교차로와 교통 수요
중심 (0,0), West/East/South 3방향과 6 OD를 검증합니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
from maps.generate_map import generate, validate_map
from src.visualization import map_figures
from IPython.display import display, Image
net=generate(); print('검증한 신호 연결:',len(validate_map(net)))
map_figures('outputs/map_preview')
for filename in ['t_junction_map.png','lane_connections.png','signal_phases.png']:
    display(Image(filename='outputs/map_preview/'+filename))

In [ ]:
from src.traffic_generator import generate_traffic
from src.utils import load_yaml
for scenario, spec in load_yaml('scenarios.yaml')['scenarios'].items():
    split='test' if spec.get('test_only') else 'train'
    seed=3000 if split=='test' else 1000
    print(generate_traffic(scenario,seed,900,split))

# 02 · 안전·강화학습·재현성 검증
실제 SUMO를 실행하는 통합 테스트도 포함합니다. 실패하면 정식 학습 전에 해결하세요.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','--junitxml=outputs/colab_test_results.xml'],check=True)

In [ ]:
import tempfile
Path('outputs').mkdir(exist_ok=True)
smoke_dir = tempfile.mkdtemp(prefix='smoke_', dir='outputs')
subprocess.run([sys.executable,'-m','scripts.smoke_test','--train','--output',smoke_dir],check=True)

In [ ]:
subprocess.run([sys.executable,'-m','scripts.benchmark_device'],check=True)

# 03 · DQN 학습과 재개
Validation의 전체 예약 차량당 관측 지연으로 best 모델을 선택합니다. 테스트 수요로 튜닝하지 마세요.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
MODE='full'
TRAIN_DIR=Path('outputs/colab_full')
DOUBLE_DQN=False
REWARD_MODE='composite'
RESUME=False
MIRROR=None
if Path('/content/drive/MyDrive').exists():
    MIRROR=Path('/content/drive/MyDrive/T_Junction_RL/runs')/TRAIN_DIR.name
print(MODE,TRAIN_DIR,MIRROR)

In [ ]:
from src.train import train
checkpoint=TRAIN_DIR/'checkpoints/training_checkpoint.pt' if RESUME else None
if checkpoint and not checkpoint.is_file() and MIRROR and (MIRROR/'checkpoints/training_checkpoint.pt').is_file():
    import shutil
    shutil.copytree(MIRROR, TRAIN_DIR, dirs_exist_ok=True)
train(MODE,TRAIN_DIR,resume=checkpoint,double=DOUBLE_DQN,reward_mode=REWARD_MODE,mirror=MIRROR)

In [ ]:
import json,pandas as pd
print((TRAIN_DIR/'training_status.json').read_text())
display(pd.read_csv(TRAIN_DIR/'logs/training.csv').tail())
print('Best 모델:',TRAIN_DIR/'checkpoints/best_model.pt')

# 04 · 同一 수요로 최종 비교
고정식·규칙식·DQN을 같은 route 파일과 SUMO seed로 평가합니다. full은 6×10×3=180회입니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
TRAIN_DIR=globals().get('TRAIN_DIR',Path('outputs/revision/seed42'))
EVAL_DIR=Path('outputs/colab_evaluation')
EVAL_MODE='full'
print('사용 모델:',TRAIN_DIR/'checkpoints/best_model.pt')

In [ ]:
from src.evaluate import evaluate
results=evaluate(TRAIN_DIR/'checkpoints/best_model.pt',EVAL_DIR,mode=EVAL_MODE,resume=(EVAL_DIR/'evaluation_plan.json').exists())
display(results.groupby('controller')[['mean_waiting_s','cohort_delay_s','completion_rate','throughput_veh_h']].mean())

In [ ]:
import pandas as pd
paired=pd.read_csv(EVAL_DIR/'scenario_comparison.csv')
display(paired[(paired.controller=='dqn')&(paired.metric=='mean_waiting_s')])

# 05 · 그래프·영상·보고서
실제 로그의 숫자와 실제 TraCI 좌표를 사용합니다. 성능이 나쁜 사례도 남깁니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
TRAIN_DIR=globals().get('TRAIN_DIR',Path('outputs/revision/seed42'))
EVAL_DIR=globals().get('EVAL_DIR',Path('outputs/example_evaluation'))
from src.visualization import plot_results,create_demo_videos
from src.export_results import generate_report
from IPython.display import display,Image,Video,Markdown
plots=plot_results(TRAIN_DIR,EVAL_DIR)
display(Image(filename=str(plots/'summary_figure.png')))

In [ ]:
CREATE_VIDEOS=True
if CREATE_VIDEOS:
    create_demo_videos(TRAIN_DIR/'checkpoints/best_model.pt',EVAL_DIR,scenario='high',seed=3000,seconds=600)
    display(Video(str(EVAL_DIR/'videos/comparison_fixed_vs_dqn.mp4'),embed=True))

In [ ]:
report=generate_report(TRAIN_DIR,EVAL_DIR)
display(Markdown(report.read_text()))

In [ ]:
if CREATE_VIDEOS:
    from src.visualization import create_worst_case_video
    path=create_worst_case_video(TRAIN_DIR/'checkpoints/best_model.pt',EVAL_DIR)
    display(Video(str(path),embed=True))

In [ ]:
ZIP_PATH = (Path('/content') if Path('/content').exists() else PROJECT_ROOT.parent)/'T_Junction_RL_results.zip'
subprocess.run([sys.executable,'-m','scripts.package_results','--output',str(ZIP_PATH)],check=True)
if Path('/content').exists():
    from google.colab import files
    files.download(str(ZIP_PATH))

# 06 · 학습된 모델 시험하기
교통량·시드·제어기를 고른 뒤 **새 실험 → 5초 진행** 또는 **끝까지 실행**을 누르세요. SUMO의 실제 차량 이동을 표시합니다. GUI 화면 없이 Colab에서 사용할 수 있습니다. 새 실험은 보고서의 최종 평가와 별도로 저장합니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
from IPython.display import display,clear_output,Video,Image
from io import BytesIO
import ipywidgets as widgets
import matplotlib.pyplot as plt
from datetime import datetime
from src.playground import Playground
if Path('/content').exists():
    from google.colab import output
    output.enable_custom_widget_manager()
model=widgets.Text(value=str(globals().get('TRAIN_DIR',Path('outputs/revision/seed42'))/'checkpoints/best_model.pt'),description='모델:',layout=widgets.Layout(width='95%'))
from src.utils import load_yaml
scenario=widgets.Dropdown(options=list(load_yaml('scenarios.yaml')['scenarios']),value='high',description='교통량:')
seed=widgets.IntText(value=3050,description='Seed:')
horizon=widgets.IntSlider(value=900,min=60,max=3600,step=60,description='시간(초):')
controller=widgets.Dropdown(options=['dqn','fixed_time','rule_based'],description='제어기:')
start=widgets.Button(description='새 실험',button_style='primary')
step=widgets.Button(description='5초 진행')
finish=widgets.Button(description='끝까지 실행')
video=widgets.Button(description='영상 생성')
canvas=widgets.Output()
session=None

def show():
    with canvas:
        clear_output(wait=True)
        figure=session.figure()
        buffer=BytesIO();figure.savefig(buffer,format='png',dpi=120)
        display(Image(data=buffer.getvalue()));plt.close('all')
        if session.last_prediction: print(session.last_prediction)
        if session.done: print('완료:',session.info['metrics'])

def begin(_):
    global session
    try:
        if session: session.close()
        path=Path('outputs/playground')/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        session=Playground(model.value,scenario.value,seed.value,horizon.value,controller.value,path)
        show()
    except Exception as e:
        with canvas: print(type(e).__name__,str(e))

def advance(_):
    if session:
        session.advance(5);show()

def complete(_):
    if session:
        finish.disabled=True
        try:
            while not session.done:
                session.advance(50);show()
        finally: finish.disabled=False

def encode(_):
    if session:
        with canvas:
            try: display(Video(str(session.video()),embed=True))
            except Exception as e: print(str(e))

start.on_click(begin);step.on_click(advance);finish.on_click(complete);video.on_click(encode)
display(model,widgets.HBox([scenario,controller]),widgets.HBox([seed,horizon]),widgets.HBox([start,step,finish,video]),canvas)